<a href="https://colab.research.google.com/github/HungHoangDinh/begin_master_reinforcement_learning/blob/main/Advantage_Actor_Critic_(A2C)_using_Robotics_Simulations_with_Panda_Gym.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create a virtual display 🔽


In [1]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [2]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

# Install dependencies 🔽


In [3]:
!pip install stable-baselines3[extra]
!pip install gymnasium
!pip install huggingface_hub
!pip install panda_gym

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 13.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873471 sha256=6d108eefa44ff5fd2f4c274e3ba875e2ecc58ab396b7b24fd09dee2188318146
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet


# Import the packages 📦


In [4]:
import os

import gymnasium as gym
import panda_gym
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

# Create the environment


In [5]:
env_id = "PandaReachDense-v3"

# Create the env
env = gym.make(env_id)

# Get the state space and action space
s_size = env.observation_space.shape
a_size = env.action_space

In [6]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

The State Space is:  None
Sample observation {'achieved_goal': array([ 6.1723776, -1.6782757,  2.0601044], dtype=float32), 'desired_goal': array([0.7318968 , 0.04629136, 9.880569  ], dtype=float32), 'observation': array([-8.92147  , -9.253842 , -4.6411805, -1.4733635, -8.364431 ,
       -5.715437 ], dtype=float32)}


In [7]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

The Action Space is:  Box(-1.0, 1.0, (3,), float32)
Action Space Sample [-0.33244216 -0.74029964  0.76781446]


In [8]:
env = make_vec_env(env_id, n_envs=4)

env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# Create the A2C Model 🤖


In [9]:
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1)

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Train the A2C agent 🏃


In [10]:
model.learn(1_000_000)

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
|    std                | 0.389    |
|    value_loss         | 0.000696 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.89     |
|    ep_rew_mean        | -0.236   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 411      |
|    iterations         | 23800    |
|    time_elapsed       | 1156     |
|    total_timesteps    | 476000   |
| train/                |          |
|    entropy_loss       | -1.17    |
|    explained_variance | 0.944    |
|    learning_rate      | 0.0007   |
|    n_updates          | 23799    |
|    policy_loss        | 0.024    |
|    std                | 0.388    |
|    value_loss         | 0.000644 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.75     |
|    ep_rew_mean  

In [11]:
# Save the model and  VecNormalize statistics when saving the agent
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")

# Evaluate the agent 📈



In [12]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load the saved statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# We need to override the render_mode
eval_env.render_mode = "rgb_array"

#  do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

Mean reward = -0.19 +/- 0.15


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [13]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
from huggingface_hub import HfApi

api = HfApi()

repo_id = "hunghd20012003/a2c-PandaReachDense-v3"

api.create_repo(repo_id=repo_id, exist_ok=True)

RepoUrl('https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='hunghd20012003/a2c-PandaReachDense-v3')

In [15]:
api.upload_file(
    path_or_fileobj="a2c-PandaReachDense-v3.zip",
    path_in_repo="a2c-PandaReachDense-v3.zip",
    repo_id=repo_id,
)

api.upload_file(
    path_or_fileobj="vec_normalize.pkl",
    path_in_repo="vec_normalize.pkl",
    repo_id=repo_id,
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  a2c-PandaReachDense-v3.zip  : 100%|##########|  114kB /  114kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  vec_normalize.pkl           : 100%|##########| 2.67kB / 2.67kB            

CommitInfo(commit_url='https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3/commit/e7a36d2586131113061c61748b92393aeaceb9fe', commit_message='Upload vec_normalize.pkl with huggingface_hub', commit_description='', oid='e7a36d2586131113061c61748b92393aeaceb9fe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='hunghd20012003/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)

In [22]:
import gymnasium as gym
import panda_gym  # QUAN TRỌNG
import imageio
import numpy as np
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

video_path = "demo.mp4"

eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

eval_env.training = False
eval_env.norm_reward = False

model = A2C.load("a2c-PandaReachDense-v3")

obs = eval_env.reset()
frames = []

for _ in range(500):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)
    frame = eval_env.render()
    frames.append(frame)

    if done:
        break

imageio.mimsave(video_path, frames, fps=30)

print("Video saved!")

Video saved!


In [23]:
api.upload_file(
    path_or_fileobj="demo.mp4",
    path_in_repo="demo.mp4",
    repo_id=repo_id,
)

CommitInfo(commit_url='https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3/commit/85015c5f5eeb2468b02cf62b3ab96a368705a18e', commit_message='Upload demo.mp4 with huggingface_hub', commit_description='', oid='85015c5f5eeb2468b02cf62b3ab96a368705a18e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='hunghd20012003/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)

In [24]:
readme = f"""
# A2C PandaReachDense-v3

Trained with Stable-Baselines3 (A2C)

## Results
Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}

## Demo
See demo.mp4
"""

with open("README.md", "w") as f:
    f.write(readme)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=repo_id,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/hf_api.py:9786: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


CommitInfo(commit_url='https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3/commit/01b4939d804cd8054b071a280554eb1e3a5c74a4', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='01b4939d804cd8054b071a280554eb1e3a5c74a4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hunghd20012003/a2c-PandaReachDense-v3', endpoint='https://huggingface.co', repo_type='model', repo_id='hunghd20012003/a2c-PandaReachDense-v3'), pr_revision=None, pr_num=None)